In [23]:
import json
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

In [25]:
pd.set_option('display.max_columns',None)

In [27]:
# Load JSON

def load_json(path):
    with open(path, "r") as f:
        return json.load(f)

In [29]:
def count_shortest_paths(G, source, sink, weight="weight"):

    predecessors, distances = nx.dijkstra_predecessor_and_distance(G, source, weight=weight)

    if sink not in distances:
        return 0

    # Dijkstra distances give a topological ordering of the
    # shortest-path DAG when edge weights are positive.
    nodes = sorted(distances, key=distances.get)

    path_counts = {node: 0 for node in nodes}
    path_counts[source] = 1

    for node in nodes:

        if node == source:
            continue

        path_counts[node] = sum(path_counts[pred] for pred in predecessors[node])

    return path_counts[sink]

In [31]:
# Extract one row of graph-level characteristics

def graph_characteristics(sample, dataset_name):
    
    n = sample["n_nodes"]
    m = sample["n_arcs"]

    u = sample["u"]
    v = sample["v"]
    dist = sample["dist"]

    source = sample["source"]
    sink = sample["sink"]

    # Build directed graph using original edge costs
    G = nx.DiGraph()

    for ui, vi, di in zip(u, v, dist):
        G.add_edge(ui, vi, weight=di)

    # Degree characteristics
    in_degrees = np.array([d for _, d in G.in_degree()])
    out_degrees = np.array([d for _, d in G.out_degree()])
    total_degrees = np.array([d for _, d in G.degree()])

    # Original source-sink shortest path
    try:
        sp_length = nx.shortest_path_length(G, source=source, target=sink, weight="weight")

        sp = nx.shortest_path(G, source=source, target=sink, weight="weight")

        sp_edges = len(sp) - 1

    except nx.NetworkXNoPath:
        sp_length = np.nan
        sp_edges = np.nan

    # Interdictability
    interdictable = sample.get("interdictable", [])
    
    if len(interdictable) > 0:
        n_interdictable = np.sum(interdictable)
        frac_interdictable = np.mean(interdictable)
    else:
        n_interdictable = np.nan
        frac_interdictable = np.nan

    num_shortest_paths = count_shortest_paths(G, source, sink, weight="weight")

    return {
        "dataset": dataset_name,
        "topology": sample.get("topology", dataset_name),

        "n_nodes": n,
        "n_arcs": m,
        "density": m / n,
        "graph_seed": sample.get("graph_seed", np.nan),

        "avg_in_degree": in_degrees.mean(),
        "avg_out_degree": out_degrees.mean(),
        "std_degree": total_degrees.std(),
        "max_degree": total_degrees.max(),

        "shortest_path_length": sp_length,
        "shortest_path_edges": sp_edges,

        "num_shortest_paths": num_shortest_paths,

        "n_interdictable": n_interdictable,
        "frac_interdictable": frac_interdictable,

        "attack_budget": sample.get(
            "attack_budget",
            sample.get("attack_limit", np.nan)),

        "mean_cost": np.mean(dist),
        "std_cost": np.std(dist),

        "mean_penalty": np.mean(sample["penalty"])
            if "penalty" in sample else np.nan,

        "optimal_interdicted_length": sample.get("path_length", np.nan)}

In [33]:
# Convert entire dataset

def summarize_dataset(data, dataset_name):

    rows = [graph_characteristics(sample, dataset_name) for sample in data]

    return pd.DataFrame(rows)

In [47]:
onein_train = load_json("training_data_shortest_path_onein_reps100.json")
mixed_60_train = load_json("training_data_shortest_path_mixed_60_10_reps100.json")
mixed_55_train = load_json("training_data_shortest_path_mixed_55_15_reps100.json")
filtered_train = load_json("training_data_shortest_path_filtered.json")


df_onein = summarize_dataset(onein_train, "One-In Training")
df_60 = summarize_dataset(mixed_60_train, "60/10 Mixed Training")
df_55 = summarize_dataset(mixed_55_train, "55/15 Mixed Training")
df_filtered = summarize_dataset(filtered_train, "Filtered One-In")

In [49]:
df_onein_unique = df_onein.drop_duplicates(subset=["graph_seed"])
df_60_unique = df_60.drop_duplicates(subset=["graph_seed"])
df_55_unique = df_55.drop_duplicates(subset=["graph_seed"])
df_filtered_unique = df_filtered.drop_duplicates(subset=["graph_seed"])

In [73]:
topology_metrics = ["shortest_path_length", "shortest_path_edges", "num_shortest_paths", "n_nodes", "n_arcs",]

topology_summary = (df_60_unique.groupby("topology")[topology_metrics].agg(["mean", "median", "min", "max"]))

topology_summary.round(3)

shortest_path_length                 shortest_path_edges             \
                          mean median min  max                mean median min   
topology                                                                        
geometric               30.742   27.5   1  102               6.900    6.0   1   
grid                     6.742    8.0   0   19               5.033    5.0   2   
layered                  7.383    6.0   1   25               2.483    2.0   1   
one_in                  15.507   13.0   1   64               3.876    4.0   1   
star_mesh                9.625    9.0   2   18               2.700    2.0   2   

              num_shortest_paths                n_nodes                 \
          max               mean median min max    mean median min max   
topology                                                                 
geometric  22              1.308    1.0   1   4  51.667   50.0  30  75   
grid        9              1.417    1.0   1   5  51.667   50.0  30  75   
layered     5              1.158    1.0   1   2  51.667   50.0  30  75   
one_in     14              1.171    1.0   1   5  51.667   50.0  30  75   
star_mesh   5              1.250    1.0   1   4  51.667   50.0  30  75   

           n_arcs                  
             mean median min  max  
topology                           
geometric  193.75  165.0  60  450  
grid       193.75  165.0  60  450  
layered    193.75  165.0  60  450  
one_in     193.75  165.0  60  450  
star_mesh  193.75  165.0  60  450

In [43]:
print(df_60_unique["topology"].value_counts())

topology
one_in       720
grid         120
geometric    120
star_mesh    120
layered      120
Name: count, dtype: int64


In [45]:
depth_counts = pd.crosstab(
    df_60_unique["topology"],
    pd.cut(
        df_60_unique["shortest_path_edges"],
        bins=[0, 5, 10, 15, 20, np.inf],
        labels=["1-5", "6-10", "11-15", "16-20", "21+"],
        include_lowest=True
    ),
    margins=True
)

display(depth_counts)

shortest_path_edges,1-5,6-10,11-15,16-20,21+,All
topology,,,,,,
geometric,53,43,18,5,1,120
grid,64,56,0,0,0,120
layered,120,0,0,0,0,120
one_in,586,131,3,0,0,720
star_mesh,120,0,0,0,0,120
All,943,230,21,5,1,1200


In [75]:
training_comparison = pd.concat([df_onein_unique, df_filtered_unique, df_60_unique, df_55_unique], ignore_index=True)

metrics = ["shortest_path_length", "shortest_path_edges", "num_shortest_paths", "n_nodes", "n_arcs"]

training_summary = (training_comparison.groupby("dataset")[metrics].agg(["mean", "median", "min", "max"]))

display(training_summary.round(3))

shortest_path_length                 shortest_path_edges  \
                                     mean median min  max                mean   
dataset                                                                         
55/15 Mixed Training               14.317   11.0   0  102               4.109   
60/10 Mixed Training               14.753   11.0   0  102               4.037   
Filtered One-In                    20.600   19.0   3   78               5.267   
One-In Training                    15.533   13.0   1   65               3.841   

                                    num_shortest_paths                n_nodes  \
                     median min max               mean median min max    mean   
dataset                                                                         
55/15 Mixed Training    3.0   1  22              1.241    1.0   1   5  51.667   
60/10 Mixed Training    3.0   1  22              1.216    1.0   1   5  51.667   
Filtered One-In         5.0   3  16              1.193    1.0   1   4  51.667   
One-In Training         4.0   1  14              1.174    1.0   1   5  51.667   

                                     n_arcs                  
                     median min max    mean median min  max  
dataset                                                      
55/15 Mixed Training   50.0  30  75  193.75  165.0  60  450  
60/10 Mixed Training   50.0  30  75  193.75  165.0  60  450  
Filtered One-In        50.0  30  75  193.75  165.0  60  450  
One-In Training        50.0  30  75  193.75  165.0  60  450

In [59]:
depth_bins = pd.cut(
    training_comparison["shortest_path_edges"],
    bins=[0, 5, 10, 15, 20, np.inf],
    labels=["1-5", "6-10", "11-15", "16-20", "21+"],
    include_lowest=True
)

depth_comparison = pd.crosstab(
    training_comparison["dataset"],
    depth_bins,
    margins=True
)

depth_comparison

shortest_path_edges,1-5,6-10,11-15,16-20,21+,All
dataset,,,,,,
55/15 Mixed Training,919,254,21,5,1,1200
60/10 Mixed Training,943,230,21,5,1,1200
Filtered One-In,391,201,7,1,0,600
One-In Training,992,204,4,0,0,1200
All,3245,889,53,11,2,4200
